In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
from sklearn.datasets import load_diabetes
diabetes = load_diabetes()

In [5]:

X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = diabetes.target

In [6]:
print(X)

          age       sex       bmi        bp        s1        s2        s3  \
0    0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1   -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2    0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3   -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4    0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   
..        ...       ...       ...       ...       ...       ...       ...   
437  0.041708  0.050680  0.019662  0.059744 -0.005697 -0.002566 -0.028674   
438 -0.005515  0.050680 -0.015906 -0.067642  0.049341  0.079165 -0.028674   
439  0.041708  0.050680 -0.015906  0.017293 -0.037344 -0.013840 -0.024993   
440 -0.045472 -0.044642  0.039062  0.001215  0.016318  0.015283 -0.028674   
441 -0.045472 -0.044642 -0.073030 -0.081413  0.083740  0.027809  0.173816   

           s4        s5        s6  
0   -0.002592  0.019907 -0.017646  
1  

In [7]:
print(y)

[151.  75. 141. 206. 135.  97. 138.  63. 110. 310. 101.  69. 179. 185.
 118. 171. 166. 144.  97. 168.  68.  49.  68. 245. 184. 202. 137.  85.
 131. 283. 129.  59. 341.  87.  65. 102. 265. 276. 252.  90. 100.  55.
  61.  92. 259.  53. 190. 142.  75. 142. 155. 225.  59. 104. 182. 128.
  52.  37. 170. 170.  61. 144.  52. 128.  71. 163. 150.  97. 160. 178.
  48. 270. 202. 111.  85.  42. 170. 200. 252. 113. 143.  51.  52. 210.
  65. 141.  55. 134.  42. 111.  98. 164.  48.  96.  90. 162. 150. 279.
  92.  83. 128. 102. 302. 198.  95.  53. 134. 144. 232.  81. 104.  59.
 246. 297. 258. 229. 275. 281. 179. 200. 200. 173. 180.  84. 121. 161.
  99. 109. 115. 268. 274. 158. 107.  83. 103. 272.  85. 280. 336. 281.
 118. 317. 235.  60. 174. 259. 178. 128.  96. 126. 288.  88. 292.  71.
 197. 186.  25.  84.  96. 195.  53. 217. 172. 131. 214.  59.  70. 220.
 268. 152.  47.  74. 295. 101. 151. 127. 237. 225.  81. 151. 107.  64.
 138. 185. 265. 101. 137. 143. 141.  79. 292. 178.  91. 116.  86. 122.
  72. 

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [11]:
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42)

In [12]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [13]:
print("R2 Score:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2 Score: 0.060653981041140725
MSE: 4976.797752808989


## **PrePruning**

In [14]:
pre_model = DecisionTreeRegressor(
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

In [15]:
pre_model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, min_samples_split=10,
                      random_state=42)

In [16]:
y_pred_pre = pre_model.predict(X_test)

In [18]:
print("R2 Score:", r2_score(y_test, y_pred_pre)* 100)

R2 Score: 43.416184229703525


## **HyperParameter** **Tuning**

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [2, 3, 4, 5, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_leaf_nodes': [10, 20, 30, None]
}

In [20]:

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [21]:
grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("Best CV Score:")
print(grid.best_score_)

Best Parameters:
{'max_depth': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score:
0.35160918571106714


In [23]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Test R2:", r2_score(y_test, y_pred)* 100)

Test R2: 29.494287913048524


## **Post**-**Pruning**

In [24]:
path = DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(
    X_train,
    y_train
)


In [25]:
ccp_alphas = path.ccp_alphas

print(ccp_alphas)

[0.00000000e+00 1.41643059e-03 1.41643059e-03 1.41643059e-03
 1.41643059e-03 1.41643059e-03 1.41643059e-03 1.41643059e-03
 1.41643059e-03 1.41643059e-03 1.41643059e-03 1.41643059e-03
 1.41643059e-03 1.41643059e-03 1.41643059e-03 1.41643059e-03
 1.88857413e-03 4.24929178e-03 4.24929178e-03 5.66572238e-03
 5.66572238e-03 5.66572238e-03 5.66572238e-03 5.66572238e-03
 5.66572238e-03 5.66572238e-03 5.66572238e-03 5.66572238e-03
 5.66572238e-03 5.66572238e-03 7.55429651e-03 1.14258735e-02
 1.18035883e-02 1.27478754e-02 1.27478754e-02 1.27478754e-02
 1.27478754e-02 1.27478754e-02 1.27478754e-02 1.51085930e-02
 1.91218130e-02 2.26628895e-02 2.26628895e-02 2.26628895e-02
 2.26628895e-02 2.26628895e-02 2.26628895e-02 2.26628895e-02
 2.26628895e-02 2.26628895e-02 2.26628895e-02 2.26628895e-02
 2.26628895e-02 2.31350330e-02 3.05949008e-02 3.54107649e-02
 3.54107649e-02 3.54107649e-02 3.54107649e-02 3.54107649e-02
 3.54107649e-02 3.54107649e-02 3.82436261e-02 4.72143532e-02
 5.09915014e-02 5.099150

In [26]:
models = []

for alpha in ccp_alphas:
    model = DecisionTreeRegressor(
        random_state=42,
        ccp_alpha=alpha
    )
    model.fit(X_train, y_train)
    models.append(model)

In [27]:
scores = []

for model in models:
    score = model.score(X_test, y_test)
    scores.append(score)

best_alpha = ccp_alphas[scores.index(max(scores))]

print("Best Alpha:", best_alpha)

Best Alpha: 80.57696804297389


In [28]:
post_model = DecisionTreeRegressor(
    random_state=42,
    ccp_alpha=best_alpha
)

post_model.fit(X_train, y_train)

y_pred_post = post_model.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred_post))

R2 Score: 0.45128467270717443
